# PD vs TBI Classification using Speech Analysis
## Smart and Connected Health - Complete ML Pipeline

This notebook implements the full pipeline:
1. **Feature Extraction** - 50+ speech features from audio files
2. **Feature Analysis** - PD vs TBI statistical comparison
3. **Model Training** - Random Forest, SVM, Gradient Boosting, Logistic Regression
4. **Evaluation** - Accuracy, Precision, Recall, F1, Confusion Matrix, ROC
5. **Inference** - Predict PD vs TBI from new audio

---
## Cell 1: Install Dependencies

In [ ]:
!pip install librosa praat-parselmouth scikit-learn xgboost matplotlib seaborn joblib -q
!apt-get install -y ffmpeg -qq

## Cell 2: Import Libraries

In [ ]:
import librosa
import parselmouth
from parselmouth.praat import call
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings

from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score, roc_curve
)
from sklearn.feature_selection import SelectKBest, f_classif

warnings.filterwarnings('ignore')
print('All libraries imported successfully!')

## Cell 3: Mount Google Drive (if using Drive for data)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
## PART 1: ENHANCED FEATURE EXTRACTION (50+ Features)

In [ ]:
def convert_to_wav(input_path):
    """Convert m4a/mp4/mp3 to wav (16kHz mono)."""
    output_path = input_path.rsplit('.', 1)[0] + '.wav'
    command = f'ffmpeg -i "{input_path}" -ar 16000 -ac 1 "{output_path}" -y -loglevel quiet'
    os.system(command)
    return output_path


def extract_voice_quality_features(sound):
    """Extract voice quality features using Parselmouth (Praat)."""
    features = {}

    # Pitch
    pitch = call(sound, "To Pitch", 0.0, 75, 500)
    features['pitch_mean'] = call(pitch, "Get mean", 0, 0, "Hertz")
    features['pitch_std'] = call(pitch, "Get standard deviation", 0, 0, "Hertz")
    features['pitch_min'] = call(pitch, "Get minimum", 0, 0, "Hertz", "Parabolic")
    features['pitch_max'] = call(pitch, "Get maximum", 0, 0, "Hertz", "Parabolic")

    # Jitter (3 variants)
    point_process = call(sound, "To PointProcess (periodic, cc)", 75, 500)
    features['jitter_local'] = call(point_process, "Get jitter (local)", 0, 0, 0.0001, 0.02, 1.3)
    features['jitter_rap'] = call(point_process, "Get jitter (rap)", 0, 0, 0.0001, 0.02, 1.3)
    features['jitter_ppq5'] = call(point_process, "Get jitter (ppq5)", 0, 0, 0.0001, 0.02, 1.3)

    # Shimmer (3 variants)
    features['shimmer_local'] = call([sound, point_process], "Get shimmer (local)", 0, 0, 0.0001, 0.02, 1.3, 1.6)
    features['shimmer_apq3'] = call([sound, point_process], "Get shimmer (apq3)", 0, 0, 0.0001, 0.02, 1.3, 1.6)
    features['shimmer_apq5'] = call([sound, point_process], "Get shimmer (apq5)", 0, 0, 0.0001, 0.02, 1.3, 1.6)

    # HNR
    harmonicity = call(sound, "To Harmonicity (cc)", 0.01, 75, 0.1, 1.0)
    features['hnr'] = call(harmonicity, "Get mean", 0, 0)

    # Formants (F1-F4)
    formant = call(sound, "To Formant (burg)", 0.0, 5, 5500, 0.025, 50)
    features['formant_f1'] = call(formant, "Get mean", 1, 0, 0, "Hertz")
    features['formant_f2'] = call(formant, "Get mean", 2, 0, 0, "Hertz")
    features['formant_f3'] = call(formant, "Get mean", 3, 0, 0, "Hertz")
    features['formant_f4'] = call(formant, "Get mean", 4, 0, 0, "Hertz")

    return features


def extract_spectral_features(audio, sr):
    """Extract spectral features using librosa."""
    features = {}

    # MFCCs (13)
    mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)
    mfcc_means = np.mean(mfccs, axis=1)
    for i in range(13):
        features[f'mfcc_{i+1}'] = mfcc_means[i]

    # Delta MFCCs (13)
    delta_mfccs = librosa.feature.delta(mfccs)
    delta_mfcc_means = np.mean(delta_mfccs, axis=1)
    for i in range(13):
        features[f'delta_mfcc_{i+1}'] = delta_mfcc_means[i]

    # Spectral features
    features['spectral_centroid'] = float(np.mean(librosa.feature.spectral_centroid(y=audio, sr=sr)))
    features['spectral_bandwidth'] = float(np.mean(librosa.feature.spectral_bandwidth(y=audio, sr=sr)))
    features['spectral_rolloff'] = float(np.mean(librosa.feature.spectral_rolloff(y=audio, sr=sr)))
    features['spectral_flatness'] = float(np.mean(librosa.feature.spectral_flatness(y=audio)))
    features['spectral_contrast'] = float(np.mean(librosa.feature.spectral_contrast(y=audio, sr=sr)))
    features['chroma_mean'] = float(np.mean(librosa.feature.chroma_stft(y=audio, sr=sr)))

    return features


def extract_temporal_features(audio, sr):
    """Extract temporal features using librosa."""
    features = {}

    features['duration'] = librosa.get_duration(y=audio, sr=sr)
    features['zcr'] = float(np.mean(librosa.feature.zero_crossing_rate(audio)))

    rms = librosa.feature.rms(y=audio)
    features['rms_energy'] = float(np.mean(rms))

    non_silent = librosa.effects.split(audio, top_db=25)
    num_pauses = max(0, len(non_silent) - 1)
    total_speech_time = sum([(end - start) / sr for start, end in non_silent])
    total_pause_time = features['duration'] - total_speech_time

    features['num_pauses'] = num_pauses
    features['pause_ratio'] = total_pause_time / features['duration'] if features['duration'] > 0 else 0
    features['speech_rate'] = len(non_silent) / features['duration'] if features['duration'] > 0 else 0

    tempo, _ = librosa.beat.beat_track(y=audio, sr=sr)
    features['tempo'] = float(np.atleast_1d(tempo)[0])

    return features


def extract_all_features(file_path):
    """Extract ALL 50+ speech features from an audio file."""
    if file_path.endswith(('.m4a', '.mp4', '.mp3')):
        wav_path = convert_to_wav(file_path)
    else:
        wav_path = file_path

    audio, sr = librosa.load(wav_path, sr=16000)
    sound = parselmouth.Sound(wav_path)

    features = {}
    features.update(extract_voice_quality_features(sound))
    features.update(extract_spectral_features(audio, sr))
    features.update(extract_temporal_features(audio, sr))

    return features


print('Feature extraction functions defined!')
print('Categories: Voice Quality (15) + Spectral (32) + Temporal (7) = 54 features')

---
## PART 2: Test Feature Extraction on a Single File

In [ ]:
# Upload your audio file or use a Drive path
# Option A: Upload directly
# from google.colab import files
# uploaded = files.upload()

# Option B: Use Drive path
test_file = "/content/20250319_141117_HBOT_070_Grandfather.m4a"

# Extract features
features = extract_all_features(test_file)

# Display results
print('=' * 60)
print('EXTRACTED FEATURES')
print('=' * 60)
print(f"{'Feature':<30} {'Value':>15}")
print('-' * 47)
for name, value in features.items():
    if isinstance(value, float):
        print(f'  {name:<28} {value:>15.6f}')
    else:
        print(f'  {name:<28} {str(value):>15}')
print('-' * 47)
print(f'Total features: {len(features)}')

---
## PART 3: Batch Process All Audio Files (PD & TBI Folders)

In [ ]:
def process_folder(folder_path, label):
    """Process all audio files in a folder with a given label."""
    all_data = []
    supported_ext = ('.wav', '.m4a', '.mp4', '.mp3')

    for filename in sorted(os.listdir(folder_path)):
        if filename.endswith(supported_ext):
            file_path = os.path.join(folder_path, filename)
            try:
                feat = extract_all_features(file_path)
                feat['filename'] = filename
                feat['label'] = label
                all_data.append(feat)
                print(f'  [OK] {filename}')
            except Exception as e:
                print(f'  [FAIL] {filename}: {e}')
    return all_data


# ===== SET YOUR DATA PATHS HERE =====
PD_FOLDER = "/content/drive/MyDrive/SmartHealth/audio_data/PD"
TBI_FOLDER = "/content/drive/MyDrive/SmartHealth/audio_data/TBI"

print('Processing PD files...')
pd_data = process_folder(PD_FOLDER, 'PD')

print('\nProcessing TBI files...')
tbi_data = process_folder(TBI_FOLDER, 'TBI')

# Combine into DataFrame
df = pd.DataFrame(pd_data + tbi_data)
print(f'\nTotal: {len(df)} files ({len(pd_data)} PD, {len(tbi_data)} TBI)')
print(f'Features per file: {len(df.columns) - 2}')

# Save to CSV
df.to_csv('features.csv', index=False)
print('\nSaved to features.csv')
df.head()

---
## PART 4: Feature Analysis - PD vs TBI Comparison

In [ ]:
# Load features (or use df from above)
df = pd.read_csv('features.csv')
drop_cols = ['filename', 'label']
feature_cols = [c for c in df.columns if c not in drop_cols]

# Group statistics
print('PD vs TBI Feature Comparison')
print('=' * 75)

stats = df.groupby('label')[feature_cols].agg(['mean', 'std'])
print(f"\n{'Feature':<25} {'PD Mean':>12} {'PD Std':>12} {'TBI Mean':>12} {'TBI Std':>12}")
print('-' * 75)
for feat in feature_cols:
    pd_m = stats.loc['PD', (feat, 'mean')] if 'PD' in stats.index else 0
    pd_s = stats.loc['PD', (feat, 'std')] if 'PD' in stats.index else 0
    tbi_m = stats.loc['TBI', (feat, 'mean')] if 'TBI' in stats.index else 0
    tbi_s = stats.loc['TBI', (feat, 'std')] if 'TBI' in stats.index else 0
    print(f'  {feat:<23} {pd_m:>12.4f} {pd_s:>12.4f} {tbi_m:>12.4f} {tbi_s:>12.4f}')

In [ ]:
# Top discriminating features (ANOVA F-test)
X = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(df[feature_cols].median())
le = LabelEncoder()
y = le.fit_transform(df['label'])

selector = SelectKBest(f_classif, k=min(15, len(feature_cols)))
selector.fit(X, y)

scores = pd.DataFrame({
    'feature': feature_cols,
    'f_score': selector.scores_,
    'p_value': selector.pvalues_
}).sort_values('f_score', ascending=False)

print('\nTop 15 Discriminating Features (ANOVA F-test)')
print('-' * 50)
for _, row in scores.head(15).iterrows():
    print(f"  {row['feature']:<25} F={row['f_score']:>10.4f}  p={row['p_value']:.6f}")

# Bar chart
fig, ax = plt.subplots(figsize=(10, 6))
top = scores.head(15)
ax.barh(top['feature'][::-1], top['f_score'][::-1], color='steelblue')
ax.set_xlabel('F-Statistic')
ax.set_title('Top 15 Discriminating Features: PD vs TBI')
plt.tight_layout()
plt.savefig('feature_importance_analysis.png', dpi=150)
plt.show()

---
## PART 5: Model Training & Evaluation

In [ ]:
# Prepare data
df = pd.read_csv('features.csv')
drop_cols = ['filename', 'label']
feature_cols = [c for c in df.columns if c not in drop_cols]

X = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(df[feature_cols].median())
le = LabelEncoder()
y = le.fit_transform(df['label'])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'Dataset: {X_scaled.shape[0]} samples, {X_scaled.shape[1]} features')
print(f'Label encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}')
print(f'Class distribution: {dict(zip(le.classes_, np.bincount(y)))}')

In [ ]:
# Define models
models = {
    'Random Forest': RandomForestClassifier(
        n_estimators=100, max_depth=10, random_state=42, class_weight='balanced'
    ),
    'SVM (RBF)': SVC(
        kernel='rbf', C=1.0, gamma='scale', probability=True,
        class_weight='balanced', random_state=42
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42
    ),
    'Logistic Regression': LogisticRegression(
        max_iter=1000, class_weight='balanced', random_state=42
    ),
}

# Cross-validation
n_splits = min(5, min(np.bincount(y)))
n_splits = max(2, n_splits)
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

print(f'{n_splits}-Fold Stratified Cross-Validation')
print('=' * 60)

results = {}
for name, model in models.items():
    cv_acc = cross_val_score(model, X_scaled, y, cv=skf, scoring='accuracy')
    cv_f1 = cross_val_score(model, X_scaled, y, cv=skf, scoring='f1_weighted')
    results[name] = {'acc': cv_acc.mean(), 'f1': cv_f1.mean()}
    print(f'\n{name}:')
    print(f'  Accuracy: {cv_acc.mean():.4f} (+/- {cv_acc.std():.4f})')
    print(f'  F1 Score: {cv_f1.mean():.4f} (+/- {cv_f1.std():.4f})')

In [ ]:
# Detailed evaluation on train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print('Detailed Evaluation (80/20 Split)')
print('=' * 60)

best_f1 = -1
best_model = None
best_name = None

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    print(f'\n{name}: Accuracy={acc:.4f}, F1={f1:.4f}')
    print(classification_report(y_test, y_pred, target_names=le.classes_, zero_division=0))

    if f1 > best_f1:
        best_f1 = f1
        best_model = model
        best_name = name

print(f'\nBest Model: {best_name} (F1={best_f1:.4f})')

In [ ]:
# Confusion Matrix
y_pred_best = best_model.predict(X_test)

fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(f'Confusion Matrix - {best_name}')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# ROC Curve
if hasattr(best_model, 'predict_proba'):
    y_prob = best_model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC = {auc:.4f}')
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'ROC Curve - {best_name}')
    ax.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig('roc_curve.png', dpi=150)
    plt.show()

In [ ]:
# Feature Importance (tree-based models)
# Retrain best model on full data
best_model_full = models[best_name]
best_model_full.fit(X_scaled, y)

if hasattr(best_model_full, 'feature_importances_'):
    importances = best_model_full.feature_importances_
    feat_imp = pd.DataFrame({
        'feature': feature_cols,
        'importance': importances
    }).sort_values('importance', ascending=False)

    fig, ax = plt.subplots(figsize=(10, 8))
    top20 = feat_imp.head(20)
    ax.barh(top20['feature'][::-1], top20['importance'][::-1], color='teal')
    ax.set_xlabel('Importance')
    ax.set_title(f'Top 20 Feature Importances - {best_name}')
    plt.tight_layout()
    plt.savefig('feature_importances.png', dpi=150)
    plt.show()

---
## PART 6: Save Model for Inference

In [ ]:
# Save model artifacts
joblib.dump(best_model_full, 'best_model.joblib')
joblib.dump(scaler, 'scaler.joblib')
joblib.dump(le, 'label_encoder.joblib')
joblib.dump(feature_cols, 'feature_names.joblib')

print('Model artifacts saved:')
print('  best_model.joblib')
print('  scaler.joblib')
print('  label_encoder.joblib')
print('  feature_names.joblib')

---
## PART 7: Inference - Predict from New Audio File

In [ ]:
def predict_from_audio(file_path):
    """Predict PD vs TBI from a new audio file."""
    # Load model artifacts
    model = joblib.load('best_model.joblib')
    scaler = joblib.load('scaler.joblib')
    le = joblib.load('label_encoder.joblib')
    feature_names = joblib.load('feature_names.joblib')

    # Extract features
    print(f'Extracting features from: {file_path}')
    features = extract_all_features(file_path)

    # Build feature vector in correct order
    feature_vector = []
    for fname in feature_names:
        val = features.get(fname, 0.0)
        if val is None or (isinstance(val, float) and (np.isnan(val) or np.isinf(val))):
            val = 0.0
        feature_vector.append(val)

    feature_vector = np.array(feature_vector).reshape(1, -1)
    feature_vector_scaled = scaler.transform(feature_vector)

    # Predict
    prediction = model.predict(feature_vector_scaled)[0]
    predicted_label = le.inverse_transform([prediction])[0]

    if hasattr(model, 'predict_proba'):
        probas = model.predict_proba(feature_vector_scaled)[0]
        confidence = probas.max()
        class_probas = dict(zip(le.classes_, probas))
    else:
        confidence = None
        class_probas = {}

    print('\n' + '=' * 50)
    print('PREDICTION RESULT')
    print('=' * 50)
    print(f'  File:       {os.path.basename(file_path)}')
    print(f'  Prediction: {predicted_label}')
    if confidence is not None:
        print(f'  Confidence: {confidence:.2%}')
        for cls, prob in class_probas.items():
            print(f'    P({cls}): {prob:.4f}')
    print('=' * 50)

    return predicted_label, confidence, class_probas


# Test inference
# predict_from_audio('/content/new_patient_recording.m4a')

---
## Summary

### Features Extracted (54 total)
| Category | Count | Features |
|----------|-------|----------|
| Voice Quality (Parselmouth) | 15 | Pitch (4), Jitter (3), Shimmer (3), HNR (1), Formants (4) |
| Spectral (Librosa) | 32 | MFCCs (13), Delta MFCCs (13), Centroid, Bandwidth, Rolloff, Flatness, Contrast, Chroma |
| Temporal (Librosa) | 7 | Duration, ZCR, RMS Energy, Pauses (2), Speech Rate, Tempo |

### Models Trained
| Model | Description |
|-------|-------------|
| Random Forest | Ensemble of decision trees, robust to overfitting |
| SVM (RBF) | Support Vector Machine with radial basis function kernel |
| Gradient Boosting | Sequential ensemble, strong on tabular data |
| Logistic Regression | Linear baseline, interpretable |

### Evaluation Metrics
- Stratified K-Fold Cross-Validation
- Accuracy, Precision, Recall, F1 Score
- Confusion Matrix, ROC Curve, AUC